In [ ]:
# Init environment before running a demo notebook.
import os
from resources.utils import *
init_demo()
# Reload the global vars again
from resources.utils import *

In [ ]:
# Load generated product in ZARR format
from sentineltoolbox.api import S3BucketCredentials, open_datatree
product_href = 's3://prip-rs-sentinel-3-s03olcefr/copernicus/s03olcefr/d13144f0-886a-4a3a-af08-94b5b47e8224/S03OLCEFR_20260811T065122_0359_A334_T74B.zarr'
eop = open_datatree(product_href, credentials=S3BucketCredentials(
    key=os.environ["S3_ACCESSKEY"],
    secret=os.environ["S3_SECRETKEY"],
    endpoint_url=os.environ["S3_ENDPOINT"],
    region_name=os.environ["S3_REGION"],
))

In [ ]:
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

STEP = 4  # downsampling factor, for a faster and lighter plot

lon_full = eop.measurements.longitude.values
lat_full = eop.measurements.latitude.values

# Rows corrupted by the S3-OLCI processor have longitude == latitude == 0 ("Null Island").
good_rows = ~((lon_full[:, 0] == 0) & (lat_full[:, 0] == 0))

lon = lon_full[good_rows][::STEP, ::STEP]
lat = lat_full[good_rows][::STEP, ::STEP]


def quicklook_band(band: xr.DataArray, lower_percentile: float = 2, upper_percentile: float = 98) -> np.ndarray:
    values = band.values[good_rows][::STEP, ::STEP].astype("float32")
    vmin, vmax = np.nanpercentile(values, [lower_percentile, upper_percentile])
    return np.clip((values - vmin) / (vmax - vmin), 0, 1)


rgb_array = np.stack([
    quicklook_band(eop.measurements.oa08_radiance),
    quicklook_band(eop.measurements.oa06_radiance),
    quicklook_band(eop.measurements.oa04_radiance),
], axis=2)

# Raw quicklook: no title, no axis, no frame, no margins - just the image, at its own aspect ratio.
height, width = rgb_array.shape[:2]
dpi = 100
fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)
ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(rgb_array)
ax.axis("off")
plt.show()

# Same quicklook, on a map. Center the projection on the scene.
projection = ccrs.Stereographic(central_longitude=float(np.mean(lon)), central_latitude=float(np.mean(lat)))

# Project the pixel coordinates ourselves, to frame the (diagonal) swath tightly.
xy = projection.transform_points(ccrs.PlateCarree(), lon, lat)
x, y = xy[..., 0], xy[..., 1]

fig = plt.figure(figsize=(14, 10))
ax = plt.axes(projection=projection)
ax.pcolormesh(lon, lat, rgb_array, shading="nearest", transform=ccrs.PlateCarree())
ax.set_extent([x.min(), x.max(), y.min(), y.max()], crs=projection)
ax.gridlines()  # no draw_labels: label placement crashes on some projections/extents
ax.set_title("S3 OLCI L1 EFR RGB quicklook - map projection")
fig.tight_layout()
plt.show()

In [ ]:
import rasterio
from PIL import Image
from rasterio.control import GroundControlPoint
from rasterio.crs import CRS
from rasterio.transform import from_bounds
from rasterio.warp import Resampling, reproject

dst_crs = CRS.from_proj4(projection.proj4_init)  # reuse the map's Stereographic projection

# Sparse ground control points from the swath's grid, expressed directly in the projected
# x/y (already computed for the map plot above) rather than raw lon/lat - see markdown above.
gcp_rows = np.linspace(0, lon.shape[0] - 1, 20, dtype=int)
gcp_cols = np.linspace(0, lon.shape[1] - 1, 20, dtype=int)
gcps = [
    GroundControlPoint(row=int(r), col=int(c), x=float(x[r, c]), y=float(y[r, c]))
    for r in gcp_rows
    for c in gcp_cols
]

# Destination grid: a regular raster covering the swath's bounding box in the projection's own
# (metric) coordinates, roughly at the same resolution as the (already downsampled) source.
res = max((x.max() - x.min()) / lon.shape[1], (y.max() - y.min()) / lon.shape[0])
dst_width = int((x.max() - x.min()) / res)
dst_height = int((y.max() - y.min()) / res)
dst_transform = from_bounds(x.min(), y.min(), x.max(), y.max(), dst_width, dst_height)

# NaN pixels (e.g. still-invalid ones not caught by the good_rows filter) become black.
rgb_uint8 = np.nan_to_num(rgb_array * 255, nan=0.0).astype("uint8")

cog_path = "quicklook.tif"
with rasterio.open(
    cog_path, "w",
    driver="COG",
    height=dst_height, width=dst_width,
    count=3, dtype="uint8",
    crs=dst_crs, transform=dst_transform,
    compress="deflate",
) as dst:
    reproject(
        source=np.moveaxis(rgb_uint8, 2, 0),
        destination=rasterio.band(dst, [1, 2, 3]),
        gcps=gcps,
        src_crs=dst_crs,  # gcps are already expressed in dst_crs units
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
    )
print(f"COG written to: {cog_path}")

# Same pixels, no reprojection: a plain JPEG for quick previews.
jpeg_path = "quicklook.jpg"
Image.fromarray(rgb_uint8).save(jpeg_path, quality=90)
print(f"JPEG written to: {jpeg_path}")